# Exercise 1: Extended Kalman Filter for SLAM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(123456)

# Global constants
dt = 0.1 # discretization timestep
Q = (dt**2 * 0.1) * np.eye(11) # process noise
Q[3::, 3::] = np.zeros((8,8))
R = 0.25 * np.eye(8) # observation noise
m = np.array([0, 0, 2, 8, 8, 2, 10, 10]) # ground-truth landmark positions.

### Exercise 1.1: Dynamics and Measurement Equations
Consider the discrete-time robot dynamics model:
\begin{equation*}
\begin{split}
x_{t+1} &= x_{t} + V_{t} \cos(\theta_{t}) \Delta t, \\
y_{t+1} &= y_{t} + V_{t} \sin(\theta_{t}) \Delta t \\
\theta_{t+1} &= \theta_{t} + \omega_t \Delta t,
\end{split}
\end{equation*}
where $(x,y)$ is the robot position, $\theta$ is the heading, $\Delta t$ is the discretization timestep, and $(V,\omega)$ are the speed and angular velocity commands.

In this problem, we assume the global ground truth positions of the four landmarks are unknown, along with the robot pose. However, we assume the landmarks are stationary objects in the environment, and their combined state vector is described by:
\begin{equation*}
\begin{split}
\textbf{m} = \begin{bmatrix}
m_{1,x} & m_{1,y} & m_{2,x} & m_{2,y} & m_{3,x} & m_{3,y} & m_{4,x} & m_{4,y}
\end{bmatrix}^\top.
\end{split}
\end{equation*}
As the robot navigates through its environment, it receives noisy measurements of the positions of four landmarks in the environment relative to the robot's current pose. 
The measurement for landmark $i$ is the relative position with the measurement model:
\begin{equation*}
\begin{split}
\textbf{z}_t^{i} = 
\begin{bmatrix}
    \cos(\theta_{t}) & \sin(\theta_{t}) \\
    -\sin(\theta_{t}) & \cos(\theta_{t})
\end{bmatrix}
\Big(\begin{bmatrix}
    m_{i,x} \\ m_{i,y}
\end{bmatrix} - \begin{bmatrix}
    x_t \\ y_t
\end{bmatrix}\Big).
\end{split}
\end{equation*}
The full measurement vector of all landmarks is:
\begin{equation*}
\begin{split}
\textbf{z}_t = \begin{bmatrix} {\textbf{z}_t^{1}}^\top,  {\textbf{z}_t^{2}}^\top, {\textbf{z}_t^{3}}^\top, {\textbf{z}_t^{4}}^\top \end{bmatrix}^\top.
\end{split}
\end{equation*}

Implement these models in the functions `robot_dynamics` and `robot_measurement`. Then, implement the function `state_dynamics` to define the dynamics of the SLAM state that consists of both the robot pose and the landmark positions, $\textbf{y} = [x, y, \theta, m_{1,x}, \dots, m_{4,y}]^\top$.

In [ ]:
def robot_dynamics(x: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Robot dynamics model.

    Args:
        x: robot state, [x, y, θ]
        u: robot control vector [v, omega] where v is the
           linear velocity and omega is the angular velocity

    Returns:
        Expected next robot state.
    """
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######

def robot_measurement(x: np.ndarray) -> np.ndarray:
    """
    Robot measurement model that measures the relative position of the landmarks.

    Args:
        x: combined robot + landmarks state, [x, y, θ, m_1x, m_1y, ...], size (11,)
    
    Returns:
        Expected measurement from the given SLAM state.
    """
    px = x[0]
    py = x[1]
    th = x[2]
    m = x[3:]
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######

def state_dynamics(y: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Dynamics model for the full combined state of the SLAM problem,
    which consists of the robot pose and the landmark positions.

    Args:
        y: combined robot + landmark state, [x, y, θ, m_1x, m_1y, ...], size (11,)
        u: robot control vector [v, omega] where v is the
           linear velocity and omega is the angular velocity

    Returns:
        Expected next SLAM state.
    """
    x_robot = y[:3]
    m = y[3:]
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######

### Exercise 1.2 and 1.3: Model Jacobians
Implement the functions `dynamics_jacobian` and `measurement_jacobian` to define the Jacobians of the dynamics and measurement models you implemented above.

In [ ]:
def dynamics_jacobian(y: np.ndarray, u: np.ndarray) -> np.ndarray:
    """
    Computes the Jacobian of the dynamics model with respect 
    to the SLAM state and robot input commands.
    
    Args:
        y: combined robot + landmark state, [x, y, θ, m_1x, m_1y, ...], size (11,)
        u: robot control vector [v, omega] where v is the
           linear velocity and omega is the angular velocity
    
    Returns:
        Jacobian of the dynamics model, F, shape (11,11)
    """
    v, omega = u
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######
    return F

def measurement_jacobian(y: np.ndarray) -> np.ndarray:
    """
    Computes the Jacobian of the measurement model with respect
    to the SLAM state.
    
    Args:
        y: combined robot + landmark state, [x, y, θ, m_1x, m_1y, ...], size (11,)
    
    Returns:
        Jacobian of the measurement model, H, shape (8,3)
    """
    px = y[0]
    py = y[1]
    th = y[2]
    m = y[3:]
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######
    return H

### Exercise 1.4: Extended Kalman Filter SLAM Implementation
Implement the function `ekf_slam_update` to implement the EKF update.

In [ ]:
def ekf_slam_update(
    prior_mean: np.ndarray, 
    prior_cov: np.ndarray, 
    u: np.ndarray, 
    z: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Given the prior mean and covariance, update the belief using the 
    given control input and measurements.

    Args:
        prior_mean: prior mean SLAM state, shape (11,)
        prior_cov: prior covariance matrix, shape (11,11) 
        u: control input, shape (2,)
        z: measurement array, shape (8,)

    Returns:
        Updated mean SLAM state
        Update covariance matrix
    """
    ##### YOUR CODE STARTS HERE #####
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######  
    return mean, cov

### Exercise 1.5: Simulate EKF SLAM
Run the provided code below to simulate the robot EKF SLAM performance given an open-loop control sequence and noisy measurements and dynamics.

In [ ]:
# Time horizon to simulate
tf = 15
time = np.arange(0, tf + dt, dt)

# Ground-truth states (simulated)
y = np.zeros((11, len(time)))
y[:3, 0] = [1, 1, 0]
y[3:, 0] = np.array([0, 0, 2, 8, 8, 2, 10, 10])

# Estimated states
mean_ekf = np.zeros((11, len(time)))
cov_ekf = [np.eye(11) for _ in range(len(time))]

# Initial state and state covariance estimate
init_mu_std = 0.1
mean_ekf[:, 0] = y[:,0] + np.random.multivariate_normal(np.zeros((11,)), init_mu_std * np.eye(11))

for i in range(1, len(time)):
    # Simulation
    # True robot commands
    v = 1
    omega = np.sin(time[i])
    u = np.array([v, omega])

    # True robot dynamics with noise
    w_noise = np.random.multivariate_normal(np.zeros((11,)), Q)
    y[:, i] = state_dynamics(y[:, i - 1], u) + w_noise

    # True received measurement
    v_noise = np.random.multivariate_normal(np.zeros((8,)), R)
    z = robot_measurement(y[:, i]) + v_noise

    # Estimation
    mean_ekf[:, i], cov_ekf[i] = ekf_slam_update(mean_ekf[:, i - 1], cov_ekf[i - 1], u, z)

Finally, run the code below to generate some plots to visualize the results.

In [ ]:
plt.figure(figsize=(18, 4))
titles = ['Robot X Position', 'Robot Y Position', 'Robot Orientation']
ylabels = [r'$x_r$', r'$y_r$', r'$\theta_r$']
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.title(titles[i])
    plt.plot(time, y[i, :], linewidth=2, label='True')
    plt.plot(time, mean_ekf[i, :], '.', markersize=3, label='EKF')
    plt.xlabel('Time')
    plt.ylabel(ylabels[i])
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
titles = ['Landmark X Position', 'Landmark Y Position']
ylabels = [r'$m_x$', r'$m_y$']
colors = ['r', 'b', 'g', 'k']
for i in range(2):
    plt.subplot(1, 2, i+1)
    plt.title(titles[i])
    for j in range(4):
        plt.plot(time, y[3+i+2*j, :], linewidth=2, label='True', color=colors[j])
        plt.plot(time, mean_ekf[3+i+2*j, :], '.', markersize=3, label='EKF', color=colors[j])
    plt.xlabel('Time')
    plt.ylabel(ylabels[i])
    plt.legend()
    plt.grid(True)
plt.tight_layout()
plt.show()

#### Plot Error Ellipses Along Trajectory and Landmarks

In [ ]:
def plot_error_ellipse(ax, mean, cov, alpha=1, label=None):
    # Calculate the error ellipse parameters
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:,0][::-1]))
    
    # Compute the radius of the ellipse to correspond to the desired confidence level
    chi2_val = 2.4477  # Corresponds to 95% conf. interval
    width, height = 2 * chi2_val * np.sqrt(eigenvalues)
    
    # Draw the ellipse
    ellipse = patches.Ellipse(mean, width, height, angle=angle, edgecolor='red', fc='None', lw=1, alpha=alpha, label=label)
    ax.add_patch(ellipse)

plt.figure(figsize=(10, 8))
plt.title('Estimated Robot Trajectory and Uncertainty')
plt.plot(y[0, :], y[1, :], linewidth=2, label='Ground truth robot')
plt.plot(mean_ekf[0, :], mean_ekf[1, :], color='orange', linewidth=2, label='EKF estimate')
for j in range(4):
    plt.plot(mean_ekf[3+2*j, :], mean_ekf[4+2*j, :], 'g+')
for i in range(0, len(time), 10):
    label = 'Uncertainty' if i == 0 else None
    alpha = 0.5 + 0.5 * i / len(time)
    plot_error_ellipse(plt.gca(), mean_ekf[:2, i], cov_ekf[i][:2, :2], alpha=alpha, label=label)
    for j in range(4):
        label = 'Uncertainty' if i == 0 and j == 0 else None
        start = 3+2*j
        end = 3+2*(j+1)
        plot_error_ellipse(plt.gca(), mean_ekf[start:end, i], cov_ekf[i][start:end, start:end], alpha=alpha, label=label)
    
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()